In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks

from mqt.predictor.rl.kit_baseline import TARGET_BASIS, build_translation_pass_manager, count_two_qubit_gates
from mqt.predictor.rl.predictorenv_optonly import OptOnlyPredictorEnv

In [ ]:
checkpoint = Path('/Users/patrickhopf/Code/mqt/mqt-predictor/model_optimization_ratio_alltoall_S_checkpoints/model_optimization_ratio_alltoall_S_step_24576.zip')
icse = Path('/Users/patrickhopf/Code/icse-paper-2026-qiskit-ml')
raw_dir = icse / 'data' / 'raw'
archive_tables = icse / 'data' / 'archive' / 'tables'
test_csv = archive_tables / 'test_circuits.csv'
paper_per_circuit_csv = archive_tables / 'per_circuit_gatecount.csv'
outdir = Path('/Users/patrickhopf/Code/mqt/mqt-predictor/results/rl_eval_step_24576')
outdir.mkdir(parents=True, exist_ok=True)
results_csv = outdir / 'rl_per_circuit_gatecount.csv'
comparison_csv = outdir / 'rl_vs_icse_per_circuit_gatecount.csv'
headtohead_csv = outdir / 'rl_headtohead_vs_qiskit.csv'
summary_json = outdir / 'summary.json'

print(f'checkpoint={checkpoint}')
print(f'outdir={outdir}')
print(f'test_csv={test_csv}')

In [ ]:
test_df = pd.read_csv(test_csv)
paper_df = pd.read_csv(paper_per_circuit_csv)
ids = test_df['circuit_id'].tolist()

translation_pm = build_translation_pass_manager(TARGET_BASIS)
env = OptOnlyPredictorEnv(path_training_circuits=raw_dir, max_steps=100, pass_timeout=30, max_circuit_operations=100_000)
env.log_applied_passes = False
model = MaskablePPO.load(checkpoint, env=env)

rows = []
started = time.perf_counter()
for index, circuit_id in enumerate(ids, start=1):
    qasm_path = raw_dir / f'{circuit_id}.qasm'
    row = {
        'circuit_id': circuit_id,
        'status': 'not_started',
        'error_msg': '"''"',
        'truncated': False,
        'terminated': False,
        'reward': np.nan,
        'rl_cx': np.nan,
        'rl_partial_cx': np.nan,
        'rl_ratio': np.nan,
        'n_actions': 0,
        'actions': '"''"',
        'elapsed_sec': np.nan,
    }
    circuit_started = time.perf_counter()
    try:
        obs, info = env.reset(qasm_path, seed=0)
        terminated = False
        truncated = False
        reward = 0.0
        actions: list[str] = []
        while not (terminated or truncated):
            action_masks = get_action_masks(env)
            action, _ = model.predict(obs, action_masks=action_masks, deterministic=True)
            action = int(action)
            actions.append(str(env.action_set[action].name))
            obs, reward, terminated, truncated, info = env.step(action)

        translated = translation_pm.run(env.state)
        partial_cx = float(count_two_qubit_gates(translated))
        baseline = float(env.baseline_cx)
        row.update({
            'status': env.status,
            'error_msg': env.error_msg,
            'truncated': bool(truncated),
            'terminated': bool(terminated),
            'reward': float(reward),
            'rl_partial_cx': partial_cx,
            'n_actions': len(actions),
            'actions': ';'.join(actions),
        })
        if env.status == 'ok' and terminated and not truncated:
            row['rl_cx'] = partial_cx
            row['rl_ratio'] = baseline / partial_cx if partial_cx else np.nan
    except Exception as exc:  # noqa: BLE001
        row.update({
            'status': 'exception',
            'error_msg': f'{type(exc).__name__}: {exc}'[:500],
        })
    finally:
        row['elapsed_sec'] = time.perf_counter() - circuit_started
        rows.append(row)
        if index % 10 == 0 or index == 1 or index == len(ids):
            ok_count = sum(r['status'] == 'ok' and not r['truncated'] for r in rows)
            fail_count = len(rows) - ok_count
            print(f'[{index:03d}/{len(ids)}] ok={ok_count} fail_or_truncated={fail_count} last={circuit_id} status={row["status"]} elapsed={row["elapsed_sec"]:.2f}s')
            pd.DataFrame(rows).to_csv(results_csv, index=False)

rl_df = pd.DataFrame(rows)
rl_df.to_csv(results_csv, index=False)
comparison = paper_df.merge(rl_df, on='circuit_id', how='left', validate='one_to_one')
comparison.to_csv(comparison_csv, index=False)

head_rows = []
valid = comparison.dropna(subset=['rl_cx']).copy()
for level in range(4):
    baseline_col = f'baseline_cx_O{level}'
    comparable = valid[['rl_cx', baseline_col]].dropna()
    nontrivial = comparable[comparable[baseline_col] > 0]
    rl_vals = nontrivial['rl_cx'].to_numpy()
    base_vals = nontrivial[baseline_col].to_numpy()
    wins = int((rl_vals < base_vals).sum())
    draws = int((rl_vals == base_vals).sum())
    losses = int((rl_vals > base_vals).sum())
    n = len(nontrivial)
    all_abs = comparable[baseline_col].to_numpy() - comparable['rl_cx'].to_numpy()
    rel = 1.0 - rl_vals / base_vals if n else np.array([])
    head_rows.append({
        'method': 'rl_step_24576',
        'qiskit_level': f'O{level}',
        'n_circuits': len(comparable),
        'n_circuits_nontrivial': n,
        'wins': wins,
        'draws': draws,
        'losses': losses,
        'win_rate': wins / n if n else np.nan,
        'draw_rate': draws / n if n else np.nan,
        'loss_rate': losses / n if n else np.nan,
        'mean_relative_reduction': float(np.mean(rel)) if n else np.nan,
        'median_relative_reduction': float(np.median(rel)) if n else np.nan,
        'mean_absolute_reduction': float(np.mean(all_abs)) if len(all_abs) else np.nan,
        'median_absolute_reduction': float(np.median(all_abs)) if len(all_abs) else np.nan,
    })
head_df = pd.DataFrame(head_rows)
head_df.to_csv(headtohead_csv, index=False)

status_counts = rl_df['status'].value_counts(dropna=False).to_dict()
summary = {
    'checkpoint': str(checkpoint),
    'test_circuits': len(ids),
    'valid_ok_terminal': int((rl_df['rl_cx'].notna()).sum()),
    'status_counts': {str(k): int(v) for k, v in status_counts.items()},
    'truncated_count': int(rl_df['truncated'].sum()),
    'mean_actions': float(rl_df['n_actions'].mean()),
    'median_actions': float(rl_df['n_actions'].median()),
    'total_elapsed_sec': time.perf_counter() - started,
    'outputs': {
        'rl_per_circuit_gatecount': str(results_csv),
        'rl_vs_icse_per_circuit_gatecount': str(comparison_csv),
        'rl_headtohead_vs_qiskit': str(headtohead_csv),
    },
}
summary_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print(head_df.to_csv(index=False))